<h1> Venv Setup </h1>

In [ ]:
# venv setup: setup with python 3.10 to 3.13
# navigate to the working directory of this ipynb, then run the following commands in command prompt/terminal:
# ===== Mac =====
# python3.13 -m venv .venv
# source .venv/bin/activate
# ===== Windows =====
# py -3.13 -m venv .venv
# .venv\Scripts\activate

# Then run the pip install inside the terminal/command prompt after venv activation:
# pip install -r requirements.txt

<h1> Package Imports and Configs </h1>

In [1]:
import duckdb
import yaml
import pandas as pd

from utils.ingest_datasets import IngestDatasets
from utils.validate_cols import validate_expected_cols

from data_profiling import ProfileReport

In [2]:
# config table to store required details for each dataset 
with open('datasets_config.yaml', 'r') as file:
    datasets_config = yaml.safe_load(file)

# passing in a dev key, else will hit rate limits of 5 requests/minute. I'll revoke this key after presentation lol.
# https://guide.data.gov.sg/developer-guide/api-overview/api-rate-limits public tier is 2 requests per 10s, dev tier is 4 requests per 10s
apiKey = "v2:5ea3afd72cde0be5294b0b52f5555837c8e20fcfde2539dd8da454472cbbae64:X0lfBn-jivW4TTQwO9vAL6EMhsfecxg1"

<h1> Dataset Ingestion Through REST </h1>

In [ ]:
# dataset ingestion through REST API
IngestionHelper = IngestDatasets(apiKey=apiKey)

for dataset_name,dataset_details in datasets_config.items():
    IngestionHelper.download_dataset(
        dataset_id=dataset_details["dataset_id"],
        filename=dataset_name
    )

In [ ]:
# validate if the cols we expect from each dataset is present. raise an AssertionError indicating which cols are missing from the downloaded csvs if any.
for dataset_name,_ in datasets_config.items():
    validate_expected_cols(
        source_csv=f"raw/{dataset_name}.csv",
        datasets_config=datasets_config
    )

<h1> EDA </h1>

In [3]:
# read raw csvs into duckdb relations 
raw_jan2015_dec2016 = duckdb.read_csv("raw/ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv")  
raw_mar2012_dec2014 = duckdb.read_csv("raw/ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv")  

In [4]:
raw_master_duckdb = duckdb.sql('''
    WITH appended AS (
        SELECT 
            month
            ,town
            ,flat_type
            ,block
            ,street_name
            ,storey_range
            ,floor_area_sqm
            ,flat_model
            ,lease_commence_date
            ,NULL AS remaining_lease
            ,resale_price
            ,'ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv' AS csv_source
        FROM raw_mar2012_dec2014 

        UNION ALL 

        SELECT 
            month
            ,town
            ,flat_type
            ,block
            ,street_name
            ,storey_range
            ,floor_area_sqm
            ,flat_model
            ,lease_commence_date
            ,remaining_lease
            ,resale_price
            ,'ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv' AS csv_source
        FROM raw_jan2015_dec2016
    )

    SELECT *, ROW_NUMBER() OVER () AS index FROM appended
''')

# duckdb.sql("SELECT csv_source, COUNT(1) AS records_cnt FROM raw_master_duckdb GROUP BY 1")

In [ ]:
# use open source library for EDA 
master_raw_pd_df = raw_master_duckdb.df() 
profile = ProfileReport(master_raw_pd_df, title="Profiling Report")
profile.to_notebook_iframe()

<h1> Transform & Clean Dataset </h1>

In [5]:
# add cols required for identifying cleaned records first 
prep_cols_duckdb = duckdb.sql('''
    WITH get_avg_resale_price AS (
      SELECT 
        month
        ,town
        ,flat_type
        ,AVG(resale_price) AS avg_resale_price
      FROM raw_master_duckdb
      GROUP BY 1,2,3
    )
    
    , add_avg_resale AS (
      SELECT 
        a.*
        ,b.avg_resale_price
      FROM raw_master_duckdb a
      LEFT JOIN get_avg_resale_price b
        ON a.month=b.month
        AND a.town=b.town
        AND a.flat_type=b.flat_type
    )
    
    , enriched AS (
        SELECT 
          month
          ,town
          ,flat_type
          ,block
          ,street_name
          ,storey_range
          ,floor_area_sqm
          ,flat_model
          ,lease_commence_date
          ,remaining_lease
          ,resale_price
          ,csv_source
          ,index
          ,avg_resale_price

          -- enriched columns
          ,MD5(month||town||flat_type||block||street_name||storey_range||floor_area_sqm||flat_model||lease_commence_date||IF(remaining_lease IS NULL,'none',TRY_CAST(remaining_lease AS VARCHAR))) AS composite_key
          ,DATE_ADD(DATE(lease_commence_date||'-12-31'), INTERVAL 99 YEAR) AS lease_enddate
          ,TRY_CAST(avg_resale_price AS VARCHAR) AS avg_resale_varchar

          ,LPAD(REGEXP_REPLACE(block,'[a-z][A-Z]','','g'),3,'0') AS block_cleaned
        FROM add_avg_resale
    )

    , compute_resale_identifier AS (
      SELECT 
        month
        ,town
        ,flat_type
        ,block
        ,street_name
        ,storey_range
        ,floor_area_sqm
        ,flat_model
        ,lease_commence_date
        ,remaining_lease
        ,resale_price
        ,csv_source
        ,index

        ,avg_resale_varchar
        ,DATE_DIFF('month',CURRENT_DATE,lease_enddate) AS remaining_lease_months
        ,composite_key

        ,'S'||block_cleaned||LEFT(avg_resale_varchar,2)||RIGHT(month,2)||LEFT(town,1) AS resale_identifier
      FROM enriched
    )

    , add_validation_cols AS (
      SELECT 
        -- raw cols
        month
        ,town
        ,flat_type
        ,block
        ,street_name
        ,storey_range
        ,floor_area_sqm
        ,flat_model
        ,lease_commence_date
        ,remaining_lease
        ,resale_price
        ,csv_source
        ,index

        -- intermediate cols
        ,avg_resale_varchar
        ,remaining_lease_months
        ,TRY_CAST(FLOOR(remaining_lease_months/12) AS INTEGER)||'Years '||TRY_CAST(remaining_lease_months%12 AS VARCHAR)||'Months' AS remaining_lease_cleaned
        ,composite_key
        ,resale_identifier
        ,SHA256(resale_identifier) AS hashed_resale_identifier

        -- validation cols
        ,REGEXP_MATCHES(month,'[0-9]{4}-[0-9]{2}') AS month_format_validation
        ,ROW_NUMBER() OVER (PARTITION BY composite_key ORDER BY resale_price DESC) AS composite_key_validation
        ,ROW_NUMBER() OVER (PARTITION BY resale_identifier ORDER BY resale_price DESC) AS resale_identifier_validation 
      FROM compute_resale_identifier
    )

    SELECT * FROM add_validation_cols
''')

prep_cols_duckdb.show(max_rows=50)

┌─────────┬───────────────┬───────────┬─────────┬───────────────────┬──────────────┬────────────────┬───────────────────┬─────────────────────┬─────────────────┬──────────────┬─────────────────────────────────────────────────────────────────┬───────┬────────────────────┬────────────────────────┬─────────────────────────┬──────────────────────────────────┬───────────────────┬──────────────────────────────────────────────────────────────────┬─────────────────────────┬──────────────────────────┬──────────────────────────────┐
│  month  │     town      │ flat_type │  block  │    street_name    │ storey_range │ floor_area_sqm │    flat_model     │ lease_commence_date │ remaining_lease │ resale_price │                           csv_source                            │ index │ avg_resale_varchar │ remaining_lease_months │ remaining_lease_cleaned │          composite_key           │ resale_identifier │                     hashed_resale_identifier                     │ month_format_validation │ 

<h2> Excluding Outliers on Cost Per Sqm </h2>
* Higher sqm flats will fetch higher prices, better to normalise by sqm before checking for anomalous resale prices

In [6]:
# add 3 sigma rule based on cost per sqm to exclude outliers 
exclude_outliers_duckdb = duckdb.sql('''
  WITH compute_cost_per_sqm AS (
    SELECT 
      -- raw cols
      month
      ,town
      ,flat_type
      ,block
      ,street_name
      ,storey_range
      ,floor_area_sqm
      ,flat_model
      ,lease_commence_date
      ,remaining_lease
      ,resale_price
      ,csv_source
      ,index

      -- intermediate cols
      ,avg_resale_varchar
      ,remaining_lease_months
      ,remaining_lease_cleaned
      ,composite_key
      ,resale_identifier
      ,hashed_resale_identifier
      
      -- validation cols
      ,month_format_validation
      ,composite_key_validation
      ,resale_identifier_validation

      ,resale_price/floor_area_sqm AS cost_per_sqm
    FROM prep_cols_duckdb
  )

  , compute_limits AS (
    SELECT 
      AVG(cost_per_sqm) - 3*STDDEV(cost_per_sqm) AS lower_limit
      ,AVG(cost_per_sqm) + 3*STDDEV(cost_per_sqm) AS upper_limit
    FROM compute_cost_per_sqm
  )

  , flag_outliers AS (
    SELECT 
      a.*
      
      ,b.lower_limit
      ,b.upper_limit

      ,CASE 
        WHEN a.cost_per_sqm < b.lower_limit THEN 1
        WHEN a.cost_per_sqm > b.upper_limit THEN 1
        ELSE 0
      END AS cost_per_sqm_validation
    FROM compute_cost_per_sqm a
    CROSS JOIN compute_limits b
  )

  SELECT * FROM flag_outliers
'''
)

In [7]:
duckdb.sql("SELECT cost_per_sqm_validation, COUNT(1) AS row_cnt FROM exclude_outliers_duckdb GROUP BY 1")

┌─────────────────────────┬─────────┐
│ cost_per_sqm_validation │ row_cnt │
│          int32          │  int64  │
├─────────────────────────┼─────────┤
│                       0 │   87623 │
│                       1 │    1733 │
└─────────────────────────┴─────────┘

In [9]:
cleaned_intermediate_duckdb = duckdb.sql('''
  WITH cleaned AS (
    SELECT 
      -- raw cols
      month
      ,town
      ,flat_type
      ,block
      ,street_name
      ,storey_range
      ,floor_area_sqm
      ,flat_model
      ,lease_commence_date
      ,remaining_lease
      ,resale_price
      ,csv_source
      ,index

      -- intermediate cols
      ,avg_resale_varchar
      ,remaining_lease_months
      ,remaining_lease_cleaned
      ,composite_key
      ,resale_identifier
      ,hashed_resale_identifier
      
      -- validation cols
      ,month_format_validation
      ,composite_key_validation
      ,resale_identifier_validation

      ,lower_limit
      ,upper_limit
      ,cost_per_sqm_validation
    FROM exclude_outliers_duckdb
    WHERE 
      1=1
      AND month_format_validation
      AND composite_key_validation = 1
      AND resale_identifier_validation = 1
      AND cost_per_sqm_validation = 0
  )

  SELECT * FROM cleaned
''')

cleaned_duckdb = duckdb.sql('''
  SELECT 
    -- raw cols
    month
    ,town
    ,flat_type
    ,block
    ,street_name
    ,storey_range
    ,floor_area_sqm
    ,flat_model
    ,lease_commence_date
    ,remaining_lease
    ,resale_price
    ,csv_source
    ,index
  FROM cleaned_intermediate_duckdb
''')

transformed_duckdb = duckdb.sql('''
  WITH transformed AS (
    SELECT 
      -- raw cols
      month
      ,town
      ,flat_type
      ,block
      ,street_name
      ,storey_range
      ,floor_area_sqm
      ,flat_model
      ,lease_commence_date
      ,remaining_lease
      ,resale_price
      ,csv_source
      ,index

      -- intermediate cols
      ,remaining_lease_cleaned 
      ,composite_key
      ,resale_identifier
    FROM cleaned_intermediate_duckdb
  )

  SELECT * FROM transformed
''')

transformed_w_hash_duckdb = duckdb.sql('''
  WITH transformed AS (
    SELECT 
      -- raw cols
      month
      ,town
      ,flat_type
      ,block
      ,street_name
      ,storey_range
      ,floor_area_sqm
      ,flat_model
      ,lease_commence_date
      ,remaining_lease
      ,resale_price
      ,csv_source
      ,index

      -- intermediate cols
      ,remaining_lease_cleaned 
      ,composite_key

      ,hashed_resale_identifier
    FROM cleaned_intermediate_duckdb
    )
  
    SELECT * FROM transformed
''')

failed_duckdb = duckdb.sql('''
  WITH invalid_month_format AS (
    SELECT
      index
      ,'invalid month format' AS failure_reason
      
    FROM exclude_outliers_duckdb
    WHERE 
      1=1
      AND NOT month_format_validation
  )

  , duplicate_composite_key AS (
    SELECT 
      index
      ,'duplicated composite key' AS failure_reason
      
    FROM exclude_outliers_duckdb
    WHERE 
      1=1
      AND composite_key_validation > 1
  )

  , duplicate_resale_key AS (
    SELECT 
      index
      ,'duplicated resale key' AS failure_reason
      
    FROM exclude_outliers_duckdb
    WHERE 
      1=1
      AND resale_identifier_validation > 1
    )

    , outlier_cost_per_sqm AS (
      SELECT 
        index
        ,'outlier cost per sqm' AS failure_reason
        
      FROM exclude_outliers_duckdb
      WHERE 
        1=1
        AND cost_per_sqm_validation = 1
    )

    , appended AS (
      WITH unioned AS (
        SELECT * FROM invalid_month_format
        UNION ALL 
        SELECT * FROM duplicate_composite_key
        UNION ALL
        SELECT * FROM duplicate_resale_key
        UNION ALL
        SELECT * FROM outlier_cost_per_sqm
      )

      SELECT 
        index
        ,STRING_AGG(failure_reason,', ') AS failure_reasons
      FROM unioned
      GROUP BY 1
    )

    , add_attributes_back AS (
      SELECT 
        -- raw cols
        b.month
        ,b.town
        ,b.flat_type
        ,b.block
        ,b.street_name
        ,b.storey_range
        ,b.floor_area_sqm
        ,b.flat_model
        ,b.lease_commence_date
        ,b.remaining_lease
        ,b.resale_price
        ,b.csv_source
        ,a.index
  
        -- intermediate cols
        ,b.composite_key
        ,b.resale_identifier
        ,b.hashed_resale_identifier
        ,b.cost_per_sqm

        ,a.failure_reasons
      FROM appended a
      LEFT JOIN exclude_outliers_duckdb b
        ON a.index=b.index
    )
    
    SELECT * FROM add_attributes_back
''')

In [10]:
from pathlib import Path

# setting up output write paths
Path("output/Raw").mkdir(parents=True, exist_ok=True)
Path("output/Cleaned").mkdir(parents=True, exist_ok=True)
Path("output/Transformed").mkdir(parents=True, exist_ok=True)
Path("output/Failed").mkdir(parents=True, exist_ok=True)
Path("output/Hashed").mkdir(parents=True, exist_ok=True)

In [11]:
# writing duckdb relation outputs to disk

# raw files 
raw_mar2012_dec2014.write_csv("output/Raw/ResaleFlatPricesBasedonRegistrationDateFromMar2012toDec2014.csv")
raw_jan2015_dec2016.write_csv("output/Raw/ResaleFlatPricesBasedonRegistrationDateFromJan2015toDec2016.csv")

prep_cols_duckdb.write_csv("output/Raw/raw_master.csv")

# cleaned files
cleaned_duckdb.write_csv("output/Cleaned/cleaned.csv")

# transformed files 
transformed_duckdb.write_csv("output/Transformed/transformed.csv")

# failed records
failed_duckdb.write_csv("output/Failed/failed.csv")

# transformed with hash
# resale idetifier is hashed with standard sha256 algorithm 
transformed_w_hash_duckdb.write_csv("output/Hashed/hashed.csv")

In [13]:
# check outputs - all required cols stored in prep_cols_duckdb for easier verification on exactly which raw record failed which validation tests 
temp_pdf = failed_duckdb.df()

scan_master_raw_pdf = duckdb.sql("SELECT * FROM prep_cols_duckdb WHERE resale_identifier='S5253108B'").df()